In [1]:
import pandas as pd
from f1_platform.db.connection import get_engine

engine = get_engine()
print("Connected to:", engine.url.database)

Connected to: f1_data


In [2]:
laps = pd.read_sql("SELECT * FROM bronze.raw_laps", engine)
results = pd.read_sql("SELECT * FROM bronze.raw_results", engine)
weather = pd.read_sql("SELECT * FROM bronze.raw_weather_data", engine)
telemetry = pd.read_sql("SELECT * FROM bronze.raw_telemetry", engine)

print(f"laps:      {len(laps):>8,} rows × {len(laps.columns):>3} cols")
print(f"results:   {len(results):>8,} rows × {len(results.columns):>3} cols")
print(f"weather:   {len(weather):>8,} rows × {len(weather.columns):>3} cols")
print(f"telemetry: {len(telemetry):>8,} rows × {len(telemetry.columns):>3} cols")

laps:        32,926 rows ×  35 cols
results:        607 rows ×  26 cols
weather:      4,744 rows ×  12 cols
telemetry:   19,479 rows ×  25 cols


In [3]:
laps.dtypes

Time                           int64
Driver                        object
DriverNumber                  object
LapTime                        int64
LapNumber                    float64
Stint                        float64
PitOutTime                     int64
PitInTime                      int64
Sector1Time                    int64
Sector2Time                    int64
Sector3Time                    int64
Sector1SessionTime             int64
Sector2SessionTime             int64
Sector3SessionTime             int64
SpeedI1                      float64
SpeedI2                      float64
SpeedFL                      float64
SpeedST                      float64
IsPersonalBest                object
Compound                      object
TyreLife                     float64
FreshTyre                       bool
Team                          object
LapStartTime                   int64
LapStartDate          datetime64[ns]
TrackStatus                   object
Position                     float64
D

## Bronze Laps — Type Analysis

### Issues to fix in Silver:

**Time columns stored as int64 (nanoseconds):**
- `Time`, `LapTime`, `PitOutTime`, `PitInTime`
- `Sector1Time`, `Sector2Time`, `Sector3Time`
- `Sector1SessionTime`, `Sector2SessionTime`, `Sector3SessionTime`
- `LapStartTime`
- → Convert to seconds (float)

**Wrong type:**
- `IsPersonalBest`: object → should be bool
- `DriverNumber`: object → consider int

**Sentinel values for missing data:**
- `-9223372036854775808` appears where NaN should be
- → Replace with NULL during Silver transformation

In [4]:
laps.isna().sum()

Time                     0
Driver                   0
DriverNumber             0
LapTime                  0
LapNumber                0
Stint                  446
PitOutTime               0
PitInTime                0
Sector1Time              0
Sector2Time              0
Sector3Time              0
Sector1SessionTime       0
Sector2SessionTime       0
Sector3SessionTime       0
SpeedI1                 50
SpeedI2                935
SpeedFL               1104
SpeedST                 58
IsPersonalBest          28
Compound                 0
TyreLife               531
FreshTyre                0
Team                     0
LapStartTime             0
LapStartDate             0
TrackStatus              0
Position                50
Deleted                  0
DeletedReason           28
FastF1Generated          0
IsAccurate               0
event                    0
year                     0
ingested_at              0
source                   0
dtype: int64

In [5]:
sentinel = -9223372036854775808

# How many sentinel values we have for each int column?
print(f"LapTime sentinels:     {(laps['LapTime'] == sentinel).sum():>6,}")
print(f"PitOutTime sentinels:  {(laps['PitOutTime'] == sentinel).sum():>6,}")
print(f"PitInTime sentinels:   {(laps['PitInTime'] == sentinel).sum():>6,}")
print(f"Sector1Time sentinels: {(laps['Sector1Time'] == sentinel).sum():>6,}")
print(f"Sector2Time sentinels: {(laps['Sector2Time'] == sentinel).sum():>6,}")
print(f"Sector3Time sentinels: {(laps['Sector3Time'] == sentinel).sum():>6,}")

LapTime sentinels:        479
PitOutTime sentinels:  31,851
PitInTime sentinels:   31,872
Sector1Time sentinels:    655
Sector2Time sentinels:     70
Sector3Time sentinels:     77


In [6]:
print("=== Compound (tire types) ===")
print(laps['Compound'].value_counts())
print()
print("=== Number of unique drivers ===")
print(f"Total unique drivers: {laps['Driver'].nunique()}")
print()
print("=== Top 5 drivers by lap count ===")
print(laps['Driver'].value_counts())

=== Compound (tire types) ===
Compound
HARD            13838
MEDIUM          12497
SOFT             3741
INTERMEDIATE     2347
nan               446
None               57
Name: count, dtype: int64

=== Number of unique drivers ===
Total unique drivers: 24

=== Top 5 drivers by lap count ===
Driver
RUS    1785
OCO    1741
NOR    1722
VER    1707
HAM    1703
LEC    1702
BEA    1682
ANT    1655
PIA    1624
GAS    1604
ALB    1591
STR    1574
LAW    1556
SAI    1532
HAD    1531
HUL    1522
ALO    1516
BOR    1507
TSU    1504
COL    1332
LIN     221
PER     219
DOO     218
BOT     178
Name: count, dtype: int64


In [7]:
print("=== LapNumber range ===")
print(f"Min: {laps['LapNumber'].min()}")
print(f"Max: {laps['LapNumber'].max()}")
print()
print("=== Speed ranges (km/h) ===")
print(f"SpeedI1  range: {laps['SpeedI1'].min()} - {laps['SpeedI1'].max()}")
print(f"SpeedST  range: {laps['SpeedST'].min()} - {laps['SpeedST'].max()}")
print(f"SpeedFL  range: {laps['SpeedFL'].min()} - {laps['SpeedFL'].max()}")

=== LapNumber range ===
Min: 1.0
Max: 78.0

=== Speed ranges (km/h) ===
SpeedI1  range: 31.0 - 352.0
SpeedST  range: 38.0 - 364.0
SpeedFL  range: 62.0 - 359.0


In [8]:
sentinel = -9223372036854775808
valid_lap_times = laps[laps['LapTime'] != sentinel]['LapTime']
lap_times_seconds = valid_lap_times / 1_000_000_000

print(f"Valid lap times: {len(lap_times_seconds):,} from {len(laps):,} total")
print()
print(f"Fastest lap: {lap_times_seconds.min():.3f} seconds")
print(f"Slowest lap: {lap_times_seconds.max():.3f} seconds")
print(f"Median lap: {lap_times_seconds.median():.3f} seconds")
print(f"Mean lap: {lap_times_seconds.mean():.3f} seconds")

Valid lap times: 32,447 from 32,926 total

Fastest lap: 67.924 seconds
Slowest lap: 218.679 seconds
Median lap: 92.613 seconds
Mean lap: 92.205 seconds


# Data Quality Checks

Following standard ETL profiling practices before Silver layer transformation.

## Check 1: Duplicates

In [9]:
laps_duplicated = laps.duplicated().sum()
print(f"Duplicated laps records: {laps_duplicated}")

results_duplicated = results.duplicated().sum()
print(f"Duplicated results records: {results_duplicated}")

weather_duplicated = weather.duplicated().sum()
print(f"Duplicated weather records: {weather_duplicated}")

telemetry_duplicated = telemetry.duplicated().sum()
print(f"Duplicated telemetry records: {telemetry_duplicated}")

Duplicated laps records: 0
Duplicated results records: 0
Duplicated weather records: 0
Duplicated telemetry records: 0


In [10]:
laps_business_keys = ['year', 'event', 'Driver', 'LapNumber']
laps_business_dups = laps.duplicated(subset=laps_business_keys).sum()
print(f"Business-level duplicates in laps: {laps_business_dups}")

results_business_keys = ['year', 'event', 'DriverId']
results_business_dups = results.duplicated(subset=results_business_keys).sum()
print(f"Business-level duplicates in results: {results_business_dups}")

weather_business_keys = ['year', 'event', 'Time']
weather_business_dups = weather.duplicated(subset=weather_business_keys).sum()
print(f"Business-level duplicates in weather: {weather_business_dups}")

telemetry_business_keys = ['year', 'event', 'Date', 'Time']
telemetry_business_dups = telemetry.duplicated(subset=telemetry_business_keys).sum()
print(f"Business-level duplicates in telemetry: {telemetry_business_dups}")

Business-level duplicates in laps: 0
Business-level duplicates in results: 0
Business-level duplicates in weather: 0
Business-level duplicates in telemetry: 0


In [11]:
print(telemetry.columns.tolist())

['Date', 'SessionTime', 'DriverAhead', 'DistanceToDriverAhead', 'Time', 'RPM', 'Speed', 'nGear', 'Throttle', 'Brake', 'DRS', 'Source', 'Distance', 'RelativeDistance', 'Status', 'X', 'Y', 'Z', 'driver', 'lap_number', 'lap_time_seconds', 'event', 'year', 'ingested_at', 'source']


In [12]:
telemetry = pd.read_sql("SELECT * FROM bronze.raw_telemetry", engine)
print(telemetry.columns.tolist())
print(f"Rows: {len(telemetry):,}")

['Date', 'SessionTime', 'DriverAhead', 'DistanceToDriverAhead', 'Time', 'RPM', 'Speed', 'nGear', 'Throttle', 'Brake', 'DRS', 'Source', 'Distance', 'RelativeDistance', 'Status', 'X', 'Y', 'Z', 'driver', 'lap_number', 'lap_time_seconds', 'event', 'year', 'ingested_at', 'source']
Rows: 19,479


In [13]:
print(f"Unique races: {telemetry[['year', 'event']].drop_duplicates().shape[0]}")
print(f"Unique drivers in telemetry: {telemetry['driver'].nunique()}")
print()
print("=== Drivers with fastest laps in telemetry ===")
print(telemetry['driver'].value_counts())

Unique races: 30
Unique drivers in telemetry: 8

=== Drivers with fastest laps in telemetry ===
driver
NOR    4502
PIA    3633
ANT    3624
VER    2690
HAM    2064
RUS    1748
LEC     666
ALB     552
Name: count, dtype: int64


In [14]:
laps = pd.read_sql("SELECT * FROM bronze.raw_laps", engine)
results = pd.read_sql("SELECT * FROM bronze.raw_results", engine)
weather = pd.read_sql("SELECT * FROM bronze.raw_weather_data", engine)
telemetry = pd.read_sql("SELECT * FROM bronze.raw_telemetry", engine)

print("Races per table (year, event):")
for name, df in [("laps", laps), ("results", results), ("weather", weather), ("telemetry", telemetry)]:
    races = df[['year', 'event']].drop_duplicates().shape[0]
    print(f"  {name:10s}: {len(df):>7,} rows, {races:>3} races")

telemetry_races = set(zip(telemetry['year'], telemetry['event']))
laps_races = set(zip(laps['year'], laps['event']))

only_in_telemetry = telemetry_races - laps_races
only_in_laps = laps_races - telemetry_races

print("In telemetry but not in laps:")
for year, event in sorted(only_in_telemetry):
    print(f"  {year} {event}")

print("\nIn laps but not in telemetry:")
for year, event in sorted(only_in_laps):
    print(f"  {year} {event}")    

Races per table (year, event):
  laps      :  32,926 rows,  30 races
  results   :     607 rows,  30 races
  weather   :   4,744 rows,  30 races
  telemetry :  19,479 rows,  30 races
In telemetry but not in laps:

In laps but not in telemetry:


In [15]:
print("Sample row count for Australia 2025:")
for table in ['raw_laps', 'raw_results', 'raw_weather_data', 'raw_telemetry']:
    count = pd.read_sql(
        f"SELECT COUNT(*) as cnt FROM bronze.{table} WHERE year = 2025 AND event = 'Australia'",
        engine
    )['cnt'][0]
    print(f"  {table}: {count}")

Sample row count for Australia 2025:
  raw_laps: 927
  raw_results: 20
  raw_weather_data: 178
  raw_telemetry: 631


In [16]:
laps_per_race = laps.groupby(['year', 'event']).size().reset_index(name='lap_count')
laps_per_race = laps_per_race.sort_values('lap_count')

print("=== Races with fewest laps ===")
print(laps_per_race.head(10))
print()
print("=== Races with most laps ===")
print(laps_per_race.tail(10))

=== Races with fewest laps ===
    year                     event  lap_count
7   2025        British Grand Prix        826
6   2025        Belgian Grand Prix        879
15  2025      Las Vegas Grand Prix        886
21  2025  Saudi Arabian Grand Prix        898
27  2026        Chinese Grand Prix        924
1   2025                 Australia        927
2   2025     Australian Grand Prix        927
4   2025     Azerbaijan Grand Prix        968
13  2025        Italian Grand Prix        975
17  2025          Miami Grand Prix       1005

=== Races with most laps ===
    year                      event  lap_count
23  2025         Spanish Grand Prix       1203
11  2025  Emilia Romagna Grand Prix       1207
22  2025       Singapore Grand Prix       1229
19  2025         Pre-Season Testing       1229
24  2025       São Paulo Grand Prix       1251
16  2025     Mexico City Grand Prix       1263
8   2025        Canadian Grand Prix       1349
10  2025           Dutch Grand Prix       1364
12  2025  